In [21]:
# Cell 1 — Imports
from pathlib import Path
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.io import save_comparisons, save_treatment_comparisons

In [22]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)

print(f"Config: {config_path}")

Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\notebook_config.yaml


In [23]:
roi_model, roi_cfg = load_model(config["models"], which="roi")
spike_model, spike_cfg = load_model(config["models"], which="spike")

models = {
    "roi": roi_model,
    "roi_config": roi_cfg,
    "spike": spike_model,
    "spike_config": spike_cfg,
}

runner = VideoPipelineRunner.build(config, models)

print(f"ROI model:   {type(roi_model).__name__}")
print(f"Spike model: {type(spike_model).__name__}")

ROI model:   RandomForestClassifier
Spike model: LogisticRegression


In [24]:
EXPERIMENT_ROOT = Path(r"C:\Users\mzinn1\Desktop\GCaMP6S_EX370")  # TODO: change per experiment
assert EXPERIMENT_ROOT.exists(), f"Experiment root not found: {EXPERIMENT_ROOT}"

builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(EXPERIMENT_ROOT)
print_tree(tree)

└── GCaMP6S_EX370
    ├── Week 1
    │   ├── 1-1
    │   ├── 1-2
    │   └── 1-3
    ├── Week 2
    │   └── baseline week 2
    │       ├── 2-1
    │       ├── 2-2
    │       └── 2-3
    ├── Week 3
    │   ├── 3-1
    │   ├── 3-2
    │   ├── 3-3
    │   └── 3-4
    └── Week 4
        ├── 4-1
        ├── 4-2
        └── 4-3


In [25]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
)
processor.process_tree(tree, verbose=True)


 Processing: 1-1
  Traces: 713 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 592/713 kept (83.0%)
  Spikes: 12477/48761 kept | neurons 592 -> 592
  Grouping (combined): | combined=38

 Processing: 1-2
  Traces: 888 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 657/888 kept (74.0%)
  Spikes: 12114/57729 kept | neurons 657 -> 657
  Grouping (combined): | combined=31

 Processing: 1-3
  Traces: 1089 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 696/1089 kept (63.9%)
  Spikes: 13974/60885 kept | neurons 696 -> 696
  Grouping (combined): | combined=36

 Processing: 2-1
  Traces: 1357 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 1099/1357 kept (81.0%)
  Spikes: 22212/94297 kept | neurons 1099 -> 1099
  Grouping (combined): | combined=59

 Processing: 2-2
  Traces: 423 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 233/423 kept (55.1%)
  Spikes: 4739/17618 kept | neurons 233 -> 233
  Grouping (combined): | combined=18

 Processing: 2-3
  Traces: 1543 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 1090/1543 kept (70.

In [26]:
 
sibling_tables = processor.compare_siblings(tree)

for node_path, df in sibling_tables.items():
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


Node: C:\Users\mzinn1\Desktop\GCaMP6S_EX370
 child  n_videos  n_neurons  n_groups_combined  mean_group_size_combined  median_group_size_combined  mean_group_corr_combined  mean_spikes_per_group_combined  frac_grouped  frac_ungrouped  decay_tau_seconds_mean_unweighted  half_max_width_seconds_mean_unweighted  rise_slope_hz_mean_unweighted  decay_tau_seconds_mean_weighted  half_max_width_seconds_mean_weighted  rise_slope_hz_mean_weighted  decay_tau_seconds_mean_grouped  half_max_width_seconds_mean_grouped  rise_slope_hz_mean_grouped  decay_tau_seconds_mean_ungrouped  half_max_width_seconds_mean_ungrouped  rise_slope_hz_mean_ungrouped  spike_frequency_mean_unweighted  spike_frequency_mean_weighted  spike_frequency_mean_grouped  spike_frequency_mean_ungrouped  decay_tau_seconds_var_unweighted  decay_tau_seconds_within_unweighted  decay_tau_seconds_between_unweighted  half_max_width_seconds_var_unweighted  half_max_width_seconds_within_unweighted  half_max_width_seconds_between_unweighted  

In [27]:
save_comparisons(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)

save_treatment_comparisons(tree)